# Train-fitted lexical value-selection baseline

**Question.** Can lexical similarity between a question and raw report context select a display-answer candidate above a transparent extractive baseline?

**Protocol.** Fit TF–IDF vocabulary on train questions and train raw context only. Evaluate once on dev. Do not load the official test split. Candidates are raw table-cell values, numeric text spans, or yes/no text spans from the allowlisted context. This system performs no calculation and is not a numerical-reasoning model.

In [1]:
from __future__ import annotations

import json
import re
import tomllib
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from finevidence.data import build_structured_input, load_finqa_split
from finevidence.evaluation import ComparisonStatus, compare_display_answers, parse_display_answer

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
CONFIG_PATH = PROJECT_ROOT / "configs" / "lexical_value_baseline.toml"
config = tomllib.loads(CONFIG_PATH.read_text(encoding="utf-8"))
RAW_DIR = PROJECT_ROOT / config["dataset_root"]
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

train_records = load_finqa_split(RAW_DIR / f"{config['train_split']}.json")
dev_records = load_finqa_split(RAW_DIR / f"{config['evaluation_split']}.json")
print({"train_records": len(train_records), "dev_records": len(dev_records), "test_loaded": False})

{'train_records': 6251, 'dev_records': 883, 'test_loaded': False}


## Inference-safe candidates

The candidate builder starts from the established allowlisted structured input. It never receives `answer`, `exe_ans`, program annotations, gold evidence, retrieved artifacts, or `model_input`. Each candidate retains only its raw value and source location for post-hoc evaluation.

In [2]:
NUMBER_SPAN = re.compile(r"(?<![\w.])[-+]?(?:\d+(?:,\d{3})*(?:\.\d+)?|\.\d+)%?(?![\w.])")
YES_NO_SPAN = re.compile(r"\b(?:yes|no)\b", re.IGNORECASE)


def build_candidates(record):
    source = build_structured_input(record)
    table = source["table"]
    header = [str(cell) for cell in table[0]]
    candidates = []
    for row_index, row in enumerate(table[1:], start=1):
        row_label = str(row[0])
        for column_index, cell in enumerate(row[1:], start=1):
            raw_value = str(cell)
            if not raw_value.strip():
                continue
            candidates.append(
                {
                    "prediction": raw_value,
                    "source_key": f"table_{row_index}",
                    "context": (
                        f"table row {row_index} row label {row_label} "
                        f"column {header[column_index]} value {raw_value}"
                    ),
                }
            )
    for text_index, text in enumerate(source["pre_text"] + source["post_text"]):
        for match in NUMBER_SPAN.finditer(text):
            candidates.append(
                {
                    "prediction": match.group(),
                    "source_key": f"text_{text_index}",
                    "context": text,
                }
            )
        for match in YES_NO_SPAN.finditer(text):
            candidates.append(
                {
                    "prediction": match.group(),
                    "source_key": f"text_{text_index}",
                    "context": text,
                }
            )
    return candidates


sample_candidates = build_candidates(train_records[0])
assert set(build_structured_input(train_records[0])) == {
    "id",
    "question",
    "table",
    "pre_text",
    "post_text",
}
display(pd.DataFrame(sample_candidates).head(8))

,prediction,source_key,context
0,$ 6427,table_1,table row 1 row label fair value of forward ex...
1,$ -23158 ( 23158 ),table_1,table row 1 row label fair value of forward ex...
2,$ 20132,table_2,table row 2 row label fair value of forward ex...
3,$ -9457 ( 9457 ),table_2,table row 2 row label fair value of forward ex...
4,$ -6781 ( 6781 ),table_3,table row 3 row label fair value of forward ex...
5,$ -38294 ( 38294 ),table_3,table row 3 row label fair value of forward ex...
6,2.05%,text_0,interest rate to a variable interest rate base...
7,2.05,text_0,interest rate to a variable interest rate base...


## Fit on train context only

The vectorizer learns a train-only vocabulary from train questions and candidate contexts. It does not receive answer labels. The selected configuration is fixed before this dev run and is not selected with test data.

In [3]:
train_documents = []
for record in train_records:
    source = build_structured_input(record)
    train_documents.append(source["question"])
    train_documents.extend(candidate["context"] for candidate in build_candidates(record))

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(config["ngram_min"], config["ngram_max"]),
    max_features=config["max_features"],
)
vectorizer.fit(train_documents)
print({"train_documents": len(train_documents), "vocabulary_size": len(vectorizer.vocabulary_)})

{'train_documents': 338983, 'vocabulary_size': 50000}


## Dev inference and display-answer evaluation

Each dev question ranks only candidates generated from that record's raw context. Gold answers and evidence are read only after ranking to compute metrics and a limited evidence-alignment diagnostic.

In [4]:
prediction_rows = []
for record in dev_records:
    source = build_structured_input(record)
    candidates = build_candidates(record)
    if candidates:
        candidate_matrix = vectorizer.transform([item["context"] for item in candidates])
        question_vector = vectorizer.transform([source["question"]])
        scores = question_vector @ candidate_matrix.T
        best_index = int(scores.argmax())
        best_candidate = candidates[best_index]
        prediction = best_candidate["prediction"]
        source_key = best_candidate["source_key"]
        score = float(scores[0, best_index])
    else:
        prediction, source_key, score = None, None, None

    comparison = compare_display_answers(prediction, record["qa"]["answer"])
    parsed_reference = parse_display_answer(record["qa"]["answer"])
    prediction_rows.append(
        {
            "id": record["id"],
            "question": source["question"],
            "prediction": prediction,
            "candidate_source_key": source_key,
            "candidate_score": score,
            "reference_answer": record["qa"]["answer"],
            "reference_type": (
                "empty"
                if parsed_reference.kind == "empty"
                else "numeric_percent"
                if parsed_reference.kind == "numeric" and parsed_reference.is_percent
                else parsed_reference.kind
            ),
            "comparison_status": comparison.status.value,
            "comparison_reason": comparison.reason,
            "metric_eligible": comparison.is_metric_eligible,
            "candidate_matches_gold_evidence": source_key in record["qa"]["gold_inds"]
            if source_key
            else False,
        }
    )

prediction_frame = pd.DataFrame(prediction_rows)
eligible = prediction_frame[prediction_frame["metric_eligible"]]
metrics = pd.DataFrame(
    {
        "dev_records": [len(prediction_frame)],
        "display_answer_denominator": [len(eligible)],
        "empty_reference_answers": [(~prediction_frame["metric_eligible"]).sum()],
        "correct": [(eligible["comparison_status"] == ComparisonStatus.CORRECT.value).sum()],
    }
)
metrics["display_answer_accuracy"] = metrics["correct"] / metrics["display_answer_denominator"]
display(metrics)

per_type = (
    eligible.assign(correct=eligible["comparison_status"].eq(ComparisonStatus.CORRECT.value))
    .groupby("reference_type")
    .agg(denominator=("correct", "size"), correct=("correct", "sum"))
    .assign(accuracy=lambda frame: frame["correct"] / frame["denominator"])
)
display(per_type)

,dev_records,display_answer_denominator,empty_reference_answers,correct,display_answer_accuracy
0,883,871,12,3,0.003444


,denominator,correct,accuracy
reference_type,,,
numeric,356,3,0.008427
numeric_percent,492,0,0.000000
text,23,0,0.000000


In [5]:
incorrect = eligible[eligible["comparison_status"].ne(ComparisonStatus.CORRECT.value)].copy()
incorrect["evidence_alignment_diagnostic"] = incorrect["candidate_matches_gold_evidence"].map(
    {
        True: "candidate shares a gold evidence key; answer still mismatched",
        False: "candidate does not share a gold evidence key",
    }
)
display(
    incorrect["evidence_alignment_diagnostic"]
    .value_counts()
    .rename("incorrect_predictions")
    .to_frame()
)
display(
    prediction_frame[
        [
            "id",
            "question",
            "prediction",
            "reference_answer",
            "reference_type",
            "candidate_source_key",
            "candidate_score",
            "comparison_status",
        ]
    ].head(8)
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prediction_path = OUTPUT_DIR / "lexical_value_baseline_dev_predictions.jsonl"
with prediction_path.open("w", encoding="utf-8") as handle:
    for row in prediction_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
print(prediction_path.relative_to(PROJECT_ROOT))

,incorrect_predictions
evidence_alignment_diagnostic,
candidate shares a gold evidence key; answer still mismatched,567
candidate does not share a gold evidence key,301


,id,question,prediction,reference_answer,reference_type,candidate_source_key,candidate_score,comparison_status
0,V/2008/page_17.pdf-1,what is the average payment volume per transac...,637,127.40,numeric,table_3,0.423164,incorrect
1,C/2017/page_328.pdf-1,what was the percentage cumulative total retur...,500,93.5%,numeric_percent,text_2,0.411844,incorrect
2,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe...,60,24.69%,numeric_percent,table_3,0.213709,incorrect
3,ETR/2011/page_341.pdf-3,in 2010 what was the net change in net revenue...,2010,18.6,numeric,text_2,0.267168,incorrect
4,ETR/2004/page_213.pdf-2,what are the deferred fuel cost revisions as a...,59.1,60.3%,numeric_percent,table_2,0.451061,incorrect
5,PM/2017/page_38.pdf-1,what was the change in millions of operating i...,6.4% ( 6.4 % ),688,numeric,table_3,0.159910,incorrect
6,JPM/2008/page_85.pdf-3,what was jpmorgan chase & co's common equity t...,1,10.94%,numeric_percent,text_24,0.161395,incorrect
7,PNC/2013/page_207.pdf-1,"at the end of 2014 , the notional value of der...",$ 36197,3044%,numeric_percent,table_1,0.567448,incorrect


data/processed/lexical_value_baseline_dev_predictions.jsonl


## Result and limitations

The displayed metrics are dev-only display-answer results under the fixed comparator. This is a lexical value-selection baseline, not numerical reasoning: it cannot combine values, choose a calculation, resolve units/periods, or establish that a selected source is sufficient evidence. A candidate's overlap with a gold evidence key is a limited evaluation diagnostic only; it does not prove a retrieval or reasoning cause for an error.

Next M3 experiment: add a train-fitted retrieval evaluation that reports evidence ranking separately from answer accuracy, then compare it with a constrained arithmetic candidate only after its calculation and unit assumptions are explicitly tested.